# 02. Data Preprocessing & Feature Engineering
## EconoCausal: Uplift Modeling & Causal Machine Learning

### 1. Objectives of this Notebook

In Notebook 01, we established that the Hillstrom dataset is highly suitable for causal inference (due to the Randomized Controlled Trial design) and identified extreme zero-inflation in spend alongside heterogeneity across customer segments. 

The objective of this notebook is **NOT** to build causal models, but rather to construct a **clean, modular, and reusable production-quality preprocessing pipeline**.

#### Key Goals:
1. **Load & Validate**: Verify data hygiene (missing values, duplicates, datatypes).
2. **Feature Engineering**: Perform any necessary cleaning of raw features (e.g., stripping prefixes from categories).
3. **Pipeline Construction**: Utilize `sklearn.compose.ColumnTransformer` and `sklearn.pipeline.Pipeline` to treat Numerical, Categorical, and Binary features independently and systematically.
4. **Variable Isolation**: Strictly isolate the Treatment variable ($T$) and Outcomes ($Y$) from the Covariates ($X$) to prevent data leakage in causal models.
5. **Artifact Generation**: Persist the fitted `preprocessor.pkl` using `joblib` so that Notebooks 03+ and the ultimate prediction API can reuse the exact same preprocessing logic without code duplication.

In [1]:
import os
import logging
from pathlib import Path
from typing import Tuple, List, Dict, Any

import numpy as np
import pandas as pd
import joblib

# Scikit-Learn tools for production pipelines
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

# Configure basic logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

logger.info("Libraries imported successfully.")

2026-09-24 22:15:10 [INFO] Libraries imported successfully.


### 2. Data Ingestion & Validation
Before engineering features, we must load the raw dataset and validate its integrity. Validating missing values, duplicate rows, and correct data types prevents downstream pipeline failures and ensures statistical soundness.

In [2]:
def load_and_validate_data() -> pd.DataFrame:
    """
    Loads the Hillstrom dataset from potential relative paths and performs 
    basic validation checks (duplicates, missing values).
    """
    # Flexible path resolution
    candidate_paths = [
        Path("../Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"),
        Path("Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"),
        Path("../datasets/raw/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv")
    ]
    
    dataset_path = next((p for p in candidate_paths if p.exists()), None)
    if dataset_path is None:
        raise FileNotFoundError("Hillstrom dataset CSV not found. Please verify directory structure.")
        
    logger.info(f"Loading dataset from {dataset_path}")
    df = pd.read_csv(dataset_path)
    
    # Validation Checks
    logger.info(f"Dataset Shape: {df.shape}")
    
    missing_counts = df.isnull().sum()
    if missing_counts.sum() == 0:
        logger.info("Missing Values Validation: PASSED (0 missing values)")
    else:
        logger.warning(f"Missing Values Detected:\n{missing_counts[missing_counts > 0]}")
        
    duplicate_count = df.duplicated().sum()
    logger.info(f"Duplicate Rows Found: {duplicate_count}")
    
    # Remove perfect duplicates if any exist
    if duplicate_count > 0:
        df = df.drop_duplicates().reset_index(drop=True)
        logger.info(f"Duplicates removed. New Shape: {df.shape}")
        
    return df

df_raw = load_and_validate_data()


2026-09-24 22:15:10 [INFO] Loading dataset from ..\Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv
2026-09-24 22:15:10 [INFO] Dataset Shape: (64000, 12)
2026-09-24 22:15:10 [INFO] Missing Values Validation: PASSED (0 missing values)
2026-09-24 22:15:10 [INFO] Duplicate Rows Found: 6562
2026-09-24 22:15:11 [INFO] Duplicates removed. New Shape: (57438, 12)


### 3. Target, Treatment, and Feature Separation
In causal inference, we must explicitly distinguish:
- $X$: The observable covariates/confounders used by the model to learn heterogeneity and propensity.
- $T$: The assigned intervention (treatment).
- $Y$: The observed outcomes we wish to optimize.

We strictly split these now so our Scikit-Learn preprocessing pipeline only ever sees and transforms $X$, preventing any target leakage.

In [3]:
# Define column mappings
TREATMENT_COL = "segment"
OUTCOME_COLS = ["visit", "conversion", "spend"]

def isolate_causal_components(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    """
    Splits the raw dataframe into Features (X), Treatment (T), and Outcomes (Y).
    """
    # Extract Treatment
    T = df[TREATMENT_COL].copy()
    
    # Extract Outcomes
    Y = df[OUTCOME_COLS].copy()
    
    # Extract Features (Everything not T or Y)
    drop_cols = OUTCOME_COLS + [TREATMENT_COL]
    X = df.drop(columns=drop_cols).copy()
    
    return X, T, Y

X_raw, T_raw, Y_raw = isolate_causal_components(df_raw)

logger.info(f"Features (X) shape: {X_raw.shape}")
logger.info(f"Treatment (T) shape: {T_raw.shape}")
logger.info(f"Outcomes (Y) shape: {Y_raw.shape}")

2026-09-24 22:15:11 [INFO] Features (X) shape: (57438, 8)
2026-09-24 22:15:11 [INFO] Treatment (T) shape: (57438,)
2026-09-24 22:15:11 [INFO] Outcomes (Y) shape: (57438, 3)


### 4. Feature Engineering
Before applying standard scaling and encoding, we perform specific feature engineering. 
For example, the `history_segment` column contains strings like `"1) $0 - $100"`. While OneHotEncoder handles this natively, cleaning the categories to be purely descriptive (e.g., `"$0 - $100"`) improves the readability of generated feature names and SHAP explanations later on.

In [4]:
class CategoricalCleaner(BaseEstimator, TransformerMixin):
    """
    Custom scikit-learn transformer to clean up strings in categorical columns
    before they are passed to the OneHotEncoder.
    """
    def __init__(self, columns_to_clean: List[str]):
        self.columns_to_clean = columns_to_clean
        
    def fit(self, X: pd.DataFrame, y=None):
        return self
        
    def transform(self, X: pd.DataFrame, y=None) -> pd.DataFrame:
        X_cleaned = X.copy()
        for col in self.columns_to_clean:
            if col in X_cleaned.columns:
                # Remove prefixes like '1) ', '2) ' from history_segment
                X_cleaned[col] = X_cleaned[col].astype(str).str.replace(r'^\d+\)\s*', '', regex=True)
                X_cleaned[col] = X_cleaned[col].str.strip()
        return X_cleaned

# Demonstration of the cleaner
cleaner = CategoricalCleaner(columns_to_clean=['history_segment'])
X_cleaned = cleaner.transform(X_raw)
logger.info(f"Unique categories in raw 'history_segment': {X_raw['history_segment'].unique().tolist()}")
logger.info(f"Unique categories in cleaned 'history_segment': {X_cleaned['history_segment'].unique().tolist()}")

2026-09-24 22:15:11 [INFO] Unique categories in raw 'history_segment': ['2) $100 - $200', '3) $200 - $350', '5) $500 - $750', '1) $0 - $100', '6) $750 - $1,000', '4) $350 - $500', '7) $1,000 +']
2026-09-24 22:15:11 [INFO] Unique categories in cleaned 'history_segment': ['$100 - $200', '$200 - $350', '$500 - $750', '$0 - $100', '$750 - $1,000', '$350 - $500', '$1,000 +']


### 5. Production Preprocessing Pipelines via ColumnTransformer
Using `pandas.get_dummies()` directly is an anti-pattern for production ML because it lacks statefulness—it cannot handle unseen categories gracefully during inference and it doesn't store fitted means/variances for scaling.

Instead, we use `sklearn.pipeline.Pipeline` combined with `sklearn.compose.ColumnTransformer`. This creates a robust, serialized object that applies the exact same transformations to future inference data.

**Pipeline Architecture:**
- **Numerical Columns** (`recency`, `history`): `SimpleImputer(median)` $\rightarrow$ `StandardScaler()`
- **Binary Columns** (`mens`, `womens`, `newbie`): `SimpleImputer(most_frequent)` $\rightarrow$ Passthrough (already 0/1)
- **Categorical Columns** (`zip_code`, `channel`, `history_segment`): `CategoricalCleaner` $\rightarrow$ `SimpleImputer(constant)` $\rightarrow$ `OneHotEncoder(handle_unknown='ignore')`

In [5]:
# Grouping features by semantic type
NUMERIC_FEATURES = ["recency", "history"]
BINARY_FEATURES = ["mens", "womens", "newbie"]
CATEGORICAL_FEATURES = ["zip_code", "channel", "history_segment"]

def build_preprocessing_pipeline() -> ColumnTransformer:
    """
    Constructs the ColumnTransformer containing all feature-specific pipelines.
    """
    
    # 1. Numeric Pipeline
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    # 2. Binary Pipeline
    # For binary 0/1 indicators, scaling is mathematically unnecessary for most tree-based 
    # DML causal estimators, and passthrough preserves interpretability.
    binary_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])
    
    # 3. Categorical Pipeline
    categorical_pipeline = Pipeline(steps=[
        ("cleaner", CategoricalCleaner(columns_to_clean=["history_segment"])),
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
        # Using drop='first' avoids perfect collinearity, which is highly desirable 
        # when fitting First-Stage Propensity or LinearDML outcome models.
    ])
    
    # Combine all into a ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, NUMERIC_FEATURES),
            ("bin", binary_pipeline, BINARY_FEATURES),
            ("cat", categorical_pipeline, CATEGORICAL_FEATURES)
        ],
        remainder="drop" # Discard any unmapped columns to guarantee strict control
    )
    
    return preprocessor

preprocessor_pipeline = build_preprocessing_pipeline()

### 6. Fitting the Preprocessor and Output Validation
We now fit the ColumnTransformer to our Covariate matrix $X$, apply the transformations, and extract the generated feature names to ensure traceability.

In [6]:
# Fit and transform the feature matrix
X_processed_array = preprocessor_pipeline.fit_transform(X_raw)

# Extract robust feature names from the ColumnTransformer
def get_feature_names_out(preprocessor: ColumnTransformer) -> List[str]:
    """
    Extracts clean feature names from a fitted ColumnTransformer.
    """
    feature_names = []
    
    for name, pipeline, features in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(features)
        elif name == "bin":
            feature_names.extend(features)
        elif name == "cat":
            # The OHE is the last step in the cat pipeline
            ohe_step = pipeline.named_steps["ohe"]
            # ohe_step.get_feature_names_out requires the raw categorical feature names to prepend
            cat_names = ohe_step.get_feature_names_out(features)
            feature_names.extend(cat_names)
            
    return feature_names

final_feature_names = get_feature_names_out(preprocessor_pipeline)

# Wrap back into a Pandas DataFrame for interpretability
X_processed = pd.DataFrame(X_processed_array, columns=final_feature_names, index=X_raw.index)

logger.info(f"Processed Covariate Matrix Shape: {X_processed.shape}")
logger.info("Final Feature Dictionary:")
for idx, fname in enumerate(final_feature_names):
    logger.info(f"  {idx + 1}. {fname}")

X_processed.head()

2026-09-24 22:15:11 [INFO] Processed Covariate Matrix Shape: (57438, 15)
2026-09-24 22:15:11 [INFO] Final Feature Dictionary:
2026-09-24 22:15:11 [INFO]   1. recency
2026-09-24 22:15:11 [INFO]   2. history
2026-09-24 22:15:11 [INFO]   3. mens
2026-09-24 22:15:11 [INFO]   4. womens
2026-09-24 22:15:11 [INFO]   5. newbie
2026-09-24 22:15:11 [INFO]   6. zip_code_Surburban
2026-09-24 22:15:11 [INFO]   7. zip_code_Urban
2026-09-24 22:15:11 [INFO]   8. channel_Phone
2026-09-24 22:15:11 [INFO]   9. channel_Web
2026-09-24 22:15:11 [INFO]   10. history_segment_$1,000 +
2026-09-24 22:15:11 [INFO]   11. history_segment_$100 - $200
2026-09-24 22:15:11 [INFO]   12. history_segment_$200 - $350
2026-09-24 22:15:11 [INFO]   13. history_segment_$350 - $500
2026-09-24 22:15:11 [INFO]   14. history_segment_$500 - $750
2026-09-24 22:15:11 [INFO]   15. history_segment_$750 - $1,000


,recency,history,mens,womens,newbie,zip_code_Surburban,zip_code_Urban,channel_Phone,channel_Web,"history_segment_$1,000 +",history_segment_$100 - $200,history_segment_$200 - $350,history_segment_$350 - $500,history_segment_$500 - $750,"history_segment_$750 - $1,000"
0,1.238418,-0.476946,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.096382,0.241979,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.381891,-0.329764,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.952909,1.577637,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-1.045653,-0.850969,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


### 7. Saving Artifacts (Production Serialization)
We save the completely fitted `preprocessor.pkl` to disk. This is a critical step because future causal modeling notebooks (`03_causal_problem_formulation.ipynb`, `04_propensity_score_modeling.ipynb`, etc.) will load this artifact to ensure identical data representations, eliminating train-test skew.

In [7]:
import os

# Ensure models/artifacts directory exists
ARTIFACT_DIR = Path("../models")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

preprocessor_path = ARTIFACT_DIR / "preprocessor.pkl"

# Serialize the stateful pipeline
joblib.dump(preprocessor_pipeline, preprocessor_path)
logger.info(f"Successfully saved Preprocessing Pipeline artifact to {preprocessor_path}")

2026-09-24 22:15:11 [INFO] Successfully saved Preprocessing Pipeline artifact to ..\models\preprocessor.pkl


### 8. Final Summary & Next Steps

#### Preprocessing Architecture Summary:
- **Inputs Selected**: All 8 independent covariates were mapped to precise pipelines.
- **Numerical Scaling**: `recency` and `history` were median-imputed and z-score standardized for optimal convergence of propensity/outcome generalized linear models.
- **Categorical Encoding**: `zip_code`, `channel`, and `history_segment` were dynamically cleaned (prefix removal) and passed through a stateless-safe One-Hot Encoder mapping (with `drop='first'` to prevent exact collinearity).
- **Leakage Prevention**: We completely decoupled $T$ (`segment`) and $Y$ (`visit`, `conversion`, `spend`) from the feature preprocessing topology.
- **Dimensionality**: The input dimensionality grew precisely due to one-hot encoding, producing the final robust feature matrix ready for DoWhy/EconML.

#### Why This Design is Production-Ready:
Unlike one-off scripts using `pd.get_dummies()`, this design leverages `sklearn.pipeline.Pipeline`. The serialized `preprocessor.pkl` can ingest single real-time API requests (containing unseen/missing categories) at production inference time without shape mismatches or data exceptions.

#### Preparation for Notebook 03:
In the next notebook, `03_causal_problem_formulation.ipynb`, we will import this processed Covariate matrix ($X$), the Treatment vector ($T$), and Outcomes ($Y$) to construct our **Directed Acyclic Graphs (DAG)** and formally establish causal identification using Microsoft's **DoWhy** framework.